# Lesson 05 — Convex Hull and Convexity Defects

## Why This Lesson
Convex hull is the tightest convex wrapper around a contour.
Convexity defects are the gaps between the hull and the contour.
This is how hand gesture recognition counts fingers.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Create a hand-like shape
canvas = np.zeros((400, 400), dtype=np.uint8)
# Simplified hand: palm + fingers
cv2.ellipse(canvas, (200,280), (80,100), 0, 0, 360, 255, -1)  # palm
finger_tips = [(160,80),(190,50),(220,45),(255,60),(290,100)]
for tip in finger_tips:
    cv2.ellipse(canvas, tip, (18,55), 0, 0, 360, 255, -1)

contours, _ = cv2.findContours(canvas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
c    = max(contours, key=cv2.contourArea)
hull = cv2.convexHull(c)
defects = cv2.convexityDefects(c, cv2.convexHull(c, returnPoints=False))

vis = cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)
cv2.drawContours(vis, [c],    -1, (0,255,0), 2)
cv2.drawContours(vis, [hull], -1, (0,0,255), 2)

# Draw defect points
finger_count = 0
if defects is not None:
    for d in defects:
        s, e, f, depth = d[0]
        if depth/256.0 > 20:  # significant defect = finger gap
            far = tuple(c[f][0])
            cv2.circle(vis, far, 8, (255,0,0), -1)
            finger_count += 1

cv2.putText(vis, f'Fingers: {finger_count+1}', (10, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,0), 2)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); plt.imshow(canvas, cmap='gray'); plt.title('Hand shape (binary)')
plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title('Green=contour, Red=hull, Blue=defects (finger gaps)')
plt.show()

## Key Takeaway
Convex hull = outer envelope. Defects = where contour dips inward from hull.
Large defects between fingertips = finger gaps. Count defects + 1 = finger count.